# Tutorial: accessing the IFN_gamma hotspot feature set

This dataset is **224,088 cells** that fall inside **IFN_gamma Gi\* hotspots**, drawn from
**10 tissues** across **3 conditions**. The goal is **feature
selection** — finding features that distinguish the conditions.

**Conditions (the `group` column):**

| group | meaning | tissues |
|---|---|---|
| `A_RT_primary` | irradiated primary tumor | 4 |
| `B_RT_secondary` | abscopal (contralateral, not irradiated) | 2 |
| `C_noRT_secondary` | secondary tumor, no radiation | 4 |

**Feature blocks:** genes (lognorm expression), cell type, CellCharter niche, ligand–receptor
(LR) interaction scores, and a couple of spatial-context scores.

**Files** (in `data/ifng_hotspot_featureset/`):
- `ifng_hotspot_featureset.h5ad` — AnnData: `X`=genes, `obsm['lr']`=LR matrix, obs metadata. **Primary.**
- `ifng_hotspot_featuretable.parquet` — the same data as one flat wide table (cells × features).
- `feature_dictionary.csv` — every feature's block / dtype / `selection_adjacent` flag.
- `README.md` — modeling notes (read it!).

By the end you'll know how to: separate the groups, pull genes + gene names, decode the LR
matrix, **handle its NaNs correctly**, relate LR to the groups, and use every obs column.

## 1. Load the data

The `.h5ad` was written with a newer `anndata`, which stores a couple of `uns` values in a way
older versions can't read (you may see an error about `uns/log1p`). `ad.read_h5ad(path)` works
fine on the env that wrote it; the helper below is a **version-proof loader** that skips `uns`
and re-attaches only the keys we need.

In [ ]:
import h5py
import numpy as np
import pandas as pd
import scipy.sparse as sp
import anndata as ad
from anndata.experimental import read_dispatched
from pathlib import Path

DATA_DIR = Path("/Users/janzules/Roselab/Spatial/CAR_T/data/ifng_hotspot_featureset")

GROUP_ORDER = ["A_RT_primary", "B_RT_secondary", "C_noRT_secondary"]

def read_h5ad_robust(path):
    """Load the h5ad even on older anndata (skips null-encoded uns scalars)."""
    def cb(func, name, elem, iospec):
        if name == "/uns":
            return {}
        return func(elem)
    with h5py.File(path, "r") as f:
        adata = read_dispatched(f, cb)
        dec = lambda x: [v.decode() if isinstance(v, bytes) else v for v in x[...]]
        for k in ("lr_features", "ifn_gamma_genes"):
            if k in f["uns"]:
                adata.uns[k] = dec(f["uns"][k])
    return adata

# ad.read()
# On the env that wrote the file you can just use: adata = ad.read_h5ad(...)
adata = read_h5ad_robust(DATA_DIR / "ifng_hotspot_featureset.h5ad")
print(adata.shape, "cells x genes")
print("obs columns:", list(adata.obs.columns))
print("obsm:", list(adata.obsm.keys()))


(224088, 19059) cells x genes
obs columns: ['tissue', 'c2l_permissive', 'X_cc_2', 'IFN_gamma_score', 'IFN_gamma_score_GiZ', 'group', 'cell_id']
obsm: ['X_cc_0', 'X_cc_01', 'X_cc_012', 'X_cc_1', 'X_cc_2', 'X_scVI', 'c2l_means', 'c2l_q05', 'c2l_q95', 'lr', 'spatial']


AnnData object with n_obs × n_vars = 224088 × 19059
    obs: 'tissue', 'c2l_permissive', 'X_cc_2', 'IFN_gamma_score', 'IFN_gamma_score_GiZ', 'group', 'cell_id'
    uns: 'lr_features', 'ifn_gamma_genes'
    obsm: 'X_cc_0', 'X_cc_01', 'X_cc_012', 'X_cc_1', 'X_cc_2', 'X_scVI', 'c2l_means', 'c2l_q05', 'c2l_q95', 'lr', 'spatial'
    layers: 'counts'
    obsp: 'spatial_connectivities', 'spatial_distances'

In [26]:
adata.obs.head()

,tissue,c2l_permissive,X_cc_2,IFN_gamma_score,IFN_gamma_score_GiZ,group,cell_id
F07839_1n2_OME_cellid_000061063-1,RTCyPSCA_1_4,Erythrocyte,6,0.457858,3.512800,A_RT_primary,F07839_1n2_OME_cellid_000061063-1
F07839_1n2_OME_cellid_000061072-1,RTCyPSCA_1_4,Unknown,6,0.255543,6.952316,A_RT_primary,F07839_1n2_OME_cellid_000061072-1
F07839_1n2_OME_cellid_000061074-1,RTCyPSCA_1_4,Unknown,1,0.278147,6.116005,A_RT_primary,F07839_1n2_OME_cellid_000061074-1
F07839_1n2_OME_cellid_000061078-1,RTCyPSCA_1_4,N1_like_Neu,2,0.356990,6.961301,A_RT_primary,F07839_1n2_OME_cellid_000061078-1
F07839_1n2_OME_cellid_000061087-1,RTCyPSCA_1_4,Erythrocyte,7,0.144748,2.997721,A_RT_primary,F07839_1n2_OME_cellid_000061087-1


## 2. The obs columns (metadata)

Everything you need to slice and label the cells is in `adata.obs`:

| column | what it is | how you use it |
|---|---|---|
| `group` | **condition / TARGET** (A/B/C) | the label `y` for feature selection |
| `tissue` | biological replicate / sample | the **CV block** (group cells by this — see §3) |
| `c2l_permissive` | cell-type label (cell2location) | a categorical feature; one-hot it |
| `X_cc_2` | CellCharter spatial niche cluster | a categorical feature; one-hot it |
| `IFN_gamma_score` | per-cell IFN_gamma pathway score | spatial-context feature — **selection-adjacent** |
| `IFN_gamma_score_GiZ` | Gi\* z-score (how 'hot' the cell's neighborhood is) | spatial-context — **selection-adjacent** |
| `cell_id` | unique cell id (== `obs_names`) | join key / index |

"Selection-adjacent" = the cells were *chosen* by IFN_gamma hotspot status, so these two columns
(and the IFN_gamma genes) are partly circular — see §8.

In [11]:
print("dtypes:\n", adata.obs.dtypes, "\n")
for c in ["group", "tissue", "c2l_permissive", "X_cc_2"]:
    print(f"{c} -> {adata.obs[c].nunique()} values: {list(map(str, pd.unique(adata.obs[c])))[:8]}")
print("\nIFN_gamma_score:", adata.obs["IFN_gamma_score"].describe()[["mean", "min", "max"]].round(3).to_dict())
print("IFN_gamma_score_GiZ:", adata.obs["IFN_gamma_score_GiZ"].describe()[["mean", "min", "max"]].round(2).to_dict())
print("cell_id == obs_names:", bool((adata.obs['cell_id'].astype(str).values == adata.obs_names.values).all()))
adata.obs.head()


dtypes:
 tissue                 category
c2l_permissive         category
X_cc_2                 category
IFN_gamma_score         float64
IFN_gamma_score_GiZ     float64
group                  category
cell_id                  object
dtype: object 

group -> 3 values: ['A_RT_primary', 'B_RT_secondary', 'C_noRT_secondary']
tissue -> 10 values: ['RTCyPSCA_1_4', 'RTCyPSCA_2_4', 'CyPSCA_2_4', 'RTCyPSCA_1_3', 'RTCyPSCA_2_3', 'CyPSCA_2_3', 'CyPSCA_2_1', 'CyPSCA_2_2']
c2l_permissive -> 20 values: ['Erythrocyte', 'Unknown', 'N1_like_Neu', 'M1_like_Mac', 'M2_like_Mac', 'Classical_Mono', 'cDC', 'N2_like_Neu']
X_cc_2 -> 7 values: ['6', '1', '2', '7', '8', '0', '4']

IFN_gamma_score: {'mean': 0.371, 'min': -0.092, 'max': 1.127}
IFN_gamma_score_GiZ: {'mean': 5.61, 'min': 2.0, 'max': 26.24}
cell_id == obs_names: True


,tissue,c2l_permissive,X_cc_2,IFN_gamma_score,IFN_gamma_score_GiZ,group,cell_id
F07839_1n2_OME_cellid_000061063-1,RTCyPSCA_1_4,Erythrocyte,6,0.457858,3.512800,A_RT_primary,F07839_1n2_OME_cellid_000061063-1
F07839_1n2_OME_cellid_000061072-1,RTCyPSCA_1_4,Unknown,6,0.255543,6.952316,A_RT_primary,F07839_1n2_OME_cellid_000061072-1
F07839_1n2_OME_cellid_000061074-1,RTCyPSCA_1_4,Unknown,1,0.278147,6.116005,A_RT_primary,F07839_1n2_OME_cellid_000061074-1
F07839_1n2_OME_cellid_000061078-1,RTCyPSCA_1_4,N1_like_Neu,2,0.356990,6.961301,A_RT_primary,F07839_1n2_OME_cellid_000061078-1
F07839_1n2_OME_cellid_000061087-1,RTCyPSCA_1_4,Erythrocyte,7,0.144748,2.997721,A_RT_primary,F07839_1n2_OME_cellid_000061087-1


## 3. Separating the groups

`group` is the target. But there is a trap: only 2–4 tissues per group vs tens of
thousands of cells, and cells within a tissue are not independent. If you split cells
randomly, a model will memorize tissue/batch signatures and look great while learning nothing
about the conditions.

**Rule:** always keep tissue and cross-validate by tissue (leave-one-tissue-out). See §8.

In [12]:
# tissues per group, and the cell counts
print("tissues per group:\n", adata.obs.groupby("group", observed=True)["tissue"].nunique(), "\n")
print(pd.crosstab(adata.obs["group"], adata.obs["tissue"]).loc[GROUP_ORDER])

# subset to one group
B = adata[adata.obs["group"] == "B_RT_secondary"]
print("\ngroup B (abscopal):", B.shape)

# or a dict of per-group views
group_views = {g: adata[adata.obs["group"] == g] for g in GROUP_ORDER}
print({g: v.n_obs for g, v in group_views.items()})


tissues per group:
 group
A_RT_primary        4
B_RT_secondary      2
C_noRT_secondary    4
Name: tissue, dtype: int64 

tissue            CyPSCA_2_1  CyPSCA_2_2  CyPSCA_2_3  CyPSCA_2_4  \
group                                                              
A_RT_primary               0           0           0           0   
B_RT_secondary             0           0           0           0   
C_noRT_secondary       31686       20764       17835       24544   

tissue            RTCyPSCA_1_2  RTCyPSCA_1_3  RTCyPSCA_1_4  RTCyPSCA_1_5  \
group                                                                      
A_RT_primary             12305         21310         22306         13312   
B_RT_secondary               0             0             0             0   
C_noRT_secondary             0             0             0             0   

tissue            RTCyPSCA_2_3  RTCyPSCA_2_4  
group                                         
A_RT_primary                 0             0  
B_RT_secondary  

## 4. Genes — expression + names

`adata.X` is **log-normalized** expression (sparse, 19k genes). Gene *names* are `adata.var_names`.

**Memory tip:** never call `.toarray()` on all 19k genes at once (that's a ~35 GB dense matrix).
Subset the genes you want *first*, then densify.

In [13]:
print("n genes:", adata.n_vars, "| X is", type(adata.X).__name__, "(lognorm)")
gene_names = adata.var_names.to_list()
print("first gene names:", gene_names[:6])

# pull a SUBSET of genes as a tidy dense DataFrame
genes_of_interest = ["Cd8a", "Nkg7", "Gzmb", "Stat1", "Cxcl9", "Cxcl10", "Prf1"]
present = [g for g in genes_of_interest if g in adata.var_names]
print("found:", present)
expr = pd.DataFrame(adata[:, present].X.toarray(), index=adata.obs_names, columns=present)
display(expr.head())

# the genes that DEFINED the hotspots are stored too (these are selection-adjacent!)
ifn_genes = list(adata.uns["ifn_gamma_genes"])
print(f"\n{len(ifn_genes)} IFN_gamma genes (SELECTION-ADJACENT):", ifn_genes[:6])


n genes: 19059 | X is csr_matrix (lognorm)
first gene names: ['Xkr4', 'Rp1', 'Sox17', 'Lypla1', 'Tcea1', 'Rgs20']
found: ['Cd8a', 'Nkg7', 'Gzmb', 'Stat1', 'Cxcl9', 'Cxcl10', 'Prf1']


,Cd8a,Nkg7,Gzmb,Stat1,Cxcl9,Cxcl10,Prf1
F07839_1n2_OME_cellid_000061063-1,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000
F07839_1n2_OME_cellid_000061072-1,0.0,0.0,0.0,0.000000,0.000000,0.0,3.769834
F07839_1n2_OME_cellid_000061074-1,0.0,0.0,0.0,4.581066,3.898110,0.0,3.898110
F07839_1n2_OME_cellid_000061078-1,0.0,0.0,0.0,4.020779,0.000000,0.0,0.000000
F07839_1n2_OME_cellid_000061087-1,0.0,0.0,0.0,0.000000,2.642966,0.0,0.000000



180 IFN_gamma genes (SELECTION-ADJACENT): ['Adar', 'Apol6', 'Arid5b', 'Arl4a', 'Auts2', 'B2m']


In [44]:
pd.DataFrame(adata["F07839_1n2_OME_cellid_000061078-1", 'Gzmb'].X.toarray())#.describe()

,0
0,0.0


In [23]:
adata.X[0:5, 10:20].toarray()

array([[0.       , 0.       , 0.       , 0.       , 0.       , 0.       ,
        0.       , 0.       , 0.       , 0.       ],
       [0.       , 0.       , 0.       , 0.       , 0.       , 0.       ,
        0.       , 0.       , 3.7698343, 0.       ],
       [0.       , 0.       , 0.       , 0.       , 0.       , 0.       ,
        0.       , 0.       , 0.       , 0.       ],
       [0.       , 0.       , 0.       , 0.       , 0.       , 0.       ,
        0.       , 0.       , 0.       , 0.       ],
       [0.       , 0.       , 0.       , 0.       , 0.       , 0.       ,
        0.       , 0.       , 2.6429663, 0.       ]], dtype=float32)

## 5. The ligand–receptor (LR) matrix

The LR scores are **not** in `X`. They live in `adata.obsm['lr']` (a cells × LR-pairs array), and
the pair names are in `adata.uns['lr_features']` (same column order). Each value is a LIANA+
**local cosine** score — spatial co-expression of that ligand–receptor pair in the cell's
neighborhood. Pair names are `Ligand^Receptor`.

In [15]:
lr = pd.DataFrame(adata.obsm["lr"], index=adata.obs_names, columns=adata.uns["lr_features"])
print("LR matrix:", lr.shape, "(cells x LR pairs)")
display(lr.iloc[:3, :5])

# decode each pair into ligand / receptor
pairs = pd.DataFrame({"pair": lr.columns})
pairs[["ligand", "receptor"]] = pairs["pair"].str.split("^", expand=True)
display(pairs.head())


LR matrix: (224088, 716) (cells x LR pairs)


,Adam10^Cd44,Adam10^Axl,Adam10^Notch1,Adam10^Notch2,Adam10^Tspan17
F07839_1n2_OME_cellid_000061063-1,0.0,0.0,NaN,NaN,NaN
F07839_1n2_OME_cellid_000061072-1,0.0,0.0,NaN,NaN,NaN
F07839_1n2_OME_cellid_000061074-1,0.0,0.0,NaN,NaN,NaN


,pair,ligand,receptor
0,Adam10^Cd44,Adam10,Cd44
1,Adam10^Axl,Adam10,Axl
2,Adam10^Notch1,Adam10,Notch1
3,Adam10^Notch2,Adam10,Notch2
4,Adam10^Tspan17,Adam10,Tspan17


## 6. Dealing with the NaNs in the LR matrix

The LR matrix is ~45% NaN, and the NaNs are not random. LR pairs are computed per
tissue, and a pair is kept only if it's expressed in enough cells (`nz_prop` filter). The matrix
is the union of pairs across tissues, so a pair that wasn't computed in a tissue is **NaN for
every cell of that tissue. Within a tissue a pair is therefore all present or all absent.

Why problematic: the missingness is perfectly correlated with `tissue` with `group`.
If you impute NaN to 0 across the union, a model can "discover" pairs that are merely *present in
group B but absent in C* — a pure batch artifact.

**Fix:** keep only pairs present in **all** tissues (the intersection). That removes the
tissue-structured holes.

In [16]:
print("overall NaN fraction:", round(lr.isna().values.mean(), 3))

# a pair is all-present or all-absent within a tissue -> collapse to per-tissue presence
present_by_tissue = lr.notna().groupby(adata.obs["tissue"].values).any()   # tissues x pairs
print("\nLR pairs computed per tissue (note how much it varies):")
print(present_by_tissue.sum(axis=1).sort_values().to_string())

complete = present_by_tissue.columns[present_by_tissue.all(axis=0)]
print(f"\npairs present in ALL tissues: {len(complete)} / {lr.shape[1]}")

lr_clean = lr[complete].copy()                 # <-- use THIS for cross-condition modeling
print("residual NaNs in the intersection:", int(lr_clean.isna().values.sum()))
# (if any remain, impute, e.g. lr_clean = lr_clean.fillna(0.0))


overall NaN fraction: 0.453

LR pairs computed per tissue (note how much it varies):
RTCyPSCA_1_4    173
CyPSCA_2_3      249
CyPSCA_2_1      288
RTCyPSCA_2_3    336
CyPSCA_2_2      373
RTCyPSCA_1_2    400
RTCyPSCA_2_4    424
RTCyPSCA_1_5    526
CyPSCA_2_4      585
RTCyPSCA_1_3    658

pairs present in ALL tissues: 165 / 716


/var/folders/np/h5smkry919g1mwf1wtltfqtm0000gp/T/ipykernel_14969/1035327874.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.


residual NaNs in the intersection: 0


## 7. Using the LR data *in association with* the groups

Compare LR on the **clean (intersection)** pairs, and aggregate to the **tissue** level first —
tissues are the independent units, cells are not. A cell-level group mean would be
pseudoreplicated.

In [ ]:
# A vs B
# B vs C

In [48]:
# per-tissue mean LR (the correct unit), then attach the group of each tissue
tissue_mean = lr_clean.groupby(adata.obs["tissue"].values).mean()        # 10 tissues x clean pairs
t2g = adata.obs.drop_duplicates("tissue").set_index("tissue")["group"]
tissue_mean["group"] = t2g.reindex(tissue_mean.index).astype(str).values

# descriptive B-vs-C contrast on the tissue-level means (NOT a significance test; n=2 vs 4)
gm = tissue_mean.groupby("group")[list(complete)].mean()
diff = (gm.loc["B_RT_secondary"] - gm.loc["C_noRT_secondary"]).sort_values()
print("LR pairs most DOWN in B vs C:\n", diff.head(5).round(3).to_string())
print("\nLR pairs most UP in B vs C:\n", diff.tail(5).round(3).to_string())
print("\n(With 2 vs 4 tissues this is a trend, not significance — see README.)")


LR pairs most DOWN in B vs C:
 C3^C3ar1     -0.124
Ccl5^Ccr5    -0.110
Cd47^Sirpa   -0.106
Sirpa^Cd47   -0.106
Thbs1^Cd47   -0.099

LR pairs most UP in B vs C:
 Col18a1^Itga5    0.059
Plau^Itga5       0.061
Fn1^Itga6        0.063
Col18a1^Ptprs    0.066
Fn1^Nt5e         0.082

(With 2 vs 4 tissues this is a trend, not significance — see README.)


/var/folders/np/h5smkry919g1mwf1wtltfqtm0000gp/T/ipykernel_14969/1957523189.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.


## 8. Putting it together: build `X`, `y`, and cross-validate by tissue

Assemble the feature blocks (here: a gene subset + clean LR + one-hot categoricals), set
`y = group`, and **split by `tissue`** with `GroupKFold`. We do **not** fit a model here — that's
your job — but this is the scaffolding.

In [54]:
adata.obs['tissue'].unique().tolist()

['RTCyPSCA_1_4',
 'RTCyPSCA_2_4',
 'CyPSCA_2_4',
 'RTCyPSCA_1_3',
 'RTCyPSCA_2_3',
 'CyPSCA_2_3',
 'CyPSCA_2_1',
 'CyPSCA_2_2',
 'RTCyPSCA_1_2',
 'RTCyPSCA_1_5']

In [19]:
from sklearn.model_selection import GroupKFold

# blocks
Xg  = pd.DataFrame(adata[:, present].X.toarray(), index=adata.obs_names,
                   columns=[f"gene_{g}" for g in present])
Xlr = lr_clean.add_prefix("lr_")
Xcat = pd.get_dummies(adata.obs[["c2l_permissive", "X_cc_2"]].astype(str), prefix=["ct", "cc"])

X = pd.concat([Xg, Xlr, Xcat], axis=1)
y = adata.obs["group"].astype(str).values
tissue = adata.obs["tissue"].astype(str).values
print("X:", X.shape, "| classes:", pd.Series(y).value_counts().to_dict())

# leave-one-tissue-out style CV — NEVER split cells randomly
gkf = GroupKFold(n_splits=pd.Series(tissue).nunique())
tr, te = next(iter(gkf.split(X, y, groups=tissue)))
print("example fold -> test tissue(s):", sorted(set(tissue[te])),
      "| train tissues:", len(set(tissue[tr])))
# Remember: scale continuous blocks (gene_/lr_) for Lasso; drop selection_adjacent features.


X: (224088, 199) | classes: {'C_noRT_secondary': 94829, 'A_RT_primary': 69233, 'B_RT_secondary': 60026}
example fold -> test tissue(s): ['RTCyPSCA_2_3'] | train tissues: 9


## 9. The flat table + feature dictionary (sklearn-friendly)

If you'd rather not touch AnnData, `ifng_hotspot_featuretable.parquet` is the same data as one
wide table (`gene_*`, `lr_*`, `ct_celltype`, `cc_cluster`, `spatialctx_*`, plus `group`, `tissue`,
`cell_id`). It's 1.3 GB with 19k gene columns, so **read only the columns you need**.
`feature_dictionary.csv` documents every column, including the `selection_adjacent` leakage flag.

In [20]:
fd = pd.read_csv(DATA_DIR / "feature_dictionary.csv")
print(fd["block"].value_counts().to_string())
print("\nselection_adjacent features (consider dropping):", int(fd["selection_adjacent"].sum()))
display(fd[fd["selection_adjacent"]].head())

# read only what you need (don't pull all 19k gene_ columns)
meta_cols = ["cell_id", "group", "tissue", "ct_celltype", "cc_cluster",
             "spatialctx_IFN_gamma_score", "spatialctx_IFN_gamma_score_GiZ"]
small = pd.read_parquet(DATA_DIR / "ifng_hotspot_featuretable.parquet", columns=meta_cols)
display(small.head())
# e.g. add a few gene columns: pd.read_parquet(path, columns=meta_cols + ["gene_Cd8a","gene_Stat1"])


block
gene               19059
lr                   716
meta                   3
spatial_context        2
celltype               1
cellcharter            1

selection_adjacent features (consider dropping): 182


,feature,block,dtype,scale,selection_adjacent
5,spatialctx_IFN_gamma_score,spatial_context,float,score,True
6,spatialctx_IFN_gamma_score_GiZ,spatial_context,float,score,True
901,gene_Stat4,gene,float,lognorm,True
902,gene_Stat1,gene,float,lognorm,True
950,gene_Casp8,gene,float,lognorm,True


,cell_id,group,tissue,ct_celltype,cc_cluster,spatialctx_IFN_gamma_score,spatialctx_IFN_gamma_score_GiZ
0,F07839_1n2_OME_cellid_000061063-1,A_RT_primary,RTCyPSCA_1_4,Erythrocyte,6,0.457858,3.512800
1,F07839_1n2_OME_cellid_000061072-1,A_RT_primary,RTCyPSCA_1_4,Unknown,6,0.255543,6.952316
2,F07839_1n2_OME_cellid_000061074-1,A_RT_primary,RTCyPSCA_1_4,Unknown,1,0.278147,6.116005
3,F07839_1n2_OME_cellid_000061078-1,A_RT_primary,RTCyPSCA_1_4,N1_like_Neu,2,0.356990,6.961301
4,F07839_1n2_OME_cellid_000061087-1,A_RT_primary,RTCyPSCA_1_4,Erythrocyte,7,0.144748,2.997721


## 10. Cheat-sheet

```python
adata = read_h5ad_robust(DATA_DIR / "ifng_hotspot_featureset.h5ad")
y       = adata.obs["group"]                       # TARGET (A/B/C)
tissue  = adata.obs["tissue"]                       # CV block (GroupKFold)
genes   = adata[:, my_genes].X.toarray()            # subset THEN densify
lr      = pd.DataFrame(adata.obsm["lr"], index=adata.obs_names, columns=adata.uns["lr_features"])
keep    = lr.notna().groupby(adata.obs["tissue"].values).any().all(0)
lr_clean= lr.loc[:, keep]                            # LR pairs present in ALL tissues
```

**Do:** split by `tissue`; use `lr_clean` (intersection); scale `gene_`/`lr_` for Lasso; one-hot
`c2l_permissive`/`X_cc_2`; read the README.
**Don't:** random cell-level CV; impute the full union LR (batch leak); densify all 19k genes;
trust a model that aces cell-level CV but not leave-one-tissue-out; forget `selection_adjacent`.